Реализуйте алгоритм GAIL на среде Mountain Car. Перед этим сгенерируйте экспертные данные (из детерминированной стратегии с первой практики). Хорошей идеей будет добавить в state (observation) синус и косинус от временной метки t для лучшего обучения.

In [1]:
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
from torch.distributions.categorical import Categorical
import torch.optim as optim
import torch.nn.functional as F

from collections import deque
import random

In [2]:
states_list = []
actions_list = []

env_expert = gym.make("MountainCar-v0")
while len(states_list) < 8734:
    obs, _ = env_expert.reset()
    done = False
    t = 0
    while not done and len(states_list) < 8734:
        pos, vel = obs
        action = 2 if vel > 0 else 0  # детерминированная экспертная стратегия
        states_list.append([pos, vel, np.sin(t), np.cos(t)])
        actions_list.append(action)
        obs, _, terminated, truncated, _ = env_expert.step(action)
        done = terminated or truncated
        t += 1

states = np.array(states_list)
actions = np.array(actions_list)

states, actions = states, actions


In [3]:
obs_dim = 4
act_dim = 3
expert_obs = np.copy(states)
expert_acts = np.copy(actions)

In [4]:
class Policy(nn.Module):
    def __init__(self, obs_dim, act_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, 64), nn.ReLU(),
            nn.Linear(64, act_dim)
        )

    def forward(self, obs):
        logits = self.net(obs)
        return Categorical(logits=logits)

    def get_action(self, obs):
        dist = self.forward(obs)
        return dist.sample().item()

In [6]:
class Discriminator(nn.Module):
    def __init__(self, obs_dim, act_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim + act_dim, 64), nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, obs, act):
        act_onehot = F.one_hot(act, num_classes=3).float()
        x = torch.cat([obs, act_onehot], dim=1)
        return self.net(x)

In [7]:
class TrajectoryBuffer:
    def __init__(self):
        self.obs, self.acts, self.rews = [], [], []

    def store(self, o, a, r):
        self.obs.append(o)
        self.acts.append(a)
        self.rews.append(r)

    def get(self):
        return (
            torch.tensor(np.array(self.obs), dtype=torch.float32),
            torch.tensor(np.array(self.acts), dtype=torch.long),
            torch.tensor(np.array(self.rews), dtype=torch.float32)
        )

In [8]:
env = gym.make("MountainCar-v0", )
policy = Policy(obs_dim, act_dim)
discrim = Discriminator(obs_dim, act_dim)

policy_opt = optim.Adam(policy.parameters(), lr=1e-3)
discrim_opt = optim.Adam(discrim.parameters(), lr=1e-3)

In [11]:
for epoch in range(3000):
    buf = TrajectoryBuffer()
    obs, _ = env.reset()
    done = False
    total_reward = 0
    t = 0

    while not done:
        pos, vel = obs
        obs_aug = np.array([pos, vel, np.sin(t), np.cos(t)], dtype=np.float32)
        obs_tensor = torch.tensor(obs_aug, dtype=torch.float32).unsqueeze(0)
        action = policy.get_action(obs_tensor)
        next_obs, _, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        buf.store(obs_aug, action, 0)
        obs = next_obs
        t += 1

    agent_obs, agent_acts, _ = buf.get()

    idxs = np.random.choice(len(expert_obs), len(agent_obs), replace=False)
    exp_obs = torch.tensor(expert_obs[idxs], dtype=torch.float32)
    exp_acts = torch.tensor(expert_acts[idxs], dtype=torch.long)

    for _ in range(2):
        discrim_opt.zero_grad()

        exp_pred = discrim(exp_obs, exp_acts).squeeze()
        ag_pred = discrim(agent_obs, agent_acts).squeeze()
        disc_loss = - (torch.log(exp_pred + 1e-8).mean() + torch.log(1 - ag_pred + 1e-8).mean())
        disc_loss.backward()
        discrim_opt.step()

    with torch.no_grad():
        rewards = -torch.log(1 - discrim(agent_obs, agent_acts).squeeze() + 1e-8)

    policy_opt.zero_grad()
    dist = policy.forward(agent_obs)
    log_probs = dist.log_prob(agent_acts)
    loss = - (log_probs * rewards).mean()

    loss.backward()
    policy_opt.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}: GAIL Loss {loss.item():.3f}, Disc Loss {disc_loss.item():.3f}")


Epoch 0: GAIL Loss 0.688, Disc Loss 1.389
Epoch 10: GAIL Loss 0.733, Disc Loss 1.324
Epoch 20: GAIL Loss 0.714, Disc Loss 1.318
Epoch 30: GAIL Loss 0.675, Disc Loss 1.291
Epoch 40: GAIL Loss 0.608, Disc Loss 1.289
Epoch 50: GAIL Loss 0.582, Disc Loss 1.297
Epoch 60: GAIL Loss 0.535, Disc Loss 1.289
Epoch 70: GAIL Loss 0.529, Disc Loss 1.306
Epoch 80: GAIL Loss 0.495, Disc Loss 1.265
Epoch 90: GAIL Loss 0.486, Disc Loss 1.291
Epoch 100: GAIL Loss 0.434, Disc Loss 1.222
Epoch 110: GAIL Loss 0.456, Disc Loss 1.271
Epoch 120: GAIL Loss 0.440, Disc Loss 1.205
Epoch 130: GAIL Loss 0.442, Disc Loss 1.236
Epoch 140: GAIL Loss 0.427, Disc Loss 1.181
Epoch 150: GAIL Loss 0.403, Disc Loss 1.165
Epoch 160: GAIL Loss 0.405, Disc Loss 1.134
Epoch 170: GAIL Loss 0.360, Disc Loss 1.095
Epoch 180: GAIL Loss 0.352, Disc Loss 1.058
Epoch 190: GAIL Loss 0.362, Disc Loss 1.048
Epoch 200: GAIL Loss 0.345, Disc Loss 1.079
Epoch 210: GAIL Loss 0.364, Disc Loss 1.019
Epoch 220: GAIL Loss 0.395, Disc Loss 1.069

Протестируйте ваш алгоритм

In [13]:
for episode in range(10):
    obs, _ = env.reset()
    done = False
    total_reward = 0
    t = 0
    while not done:
        pos, vel = obs
        obs_aug = np.array([pos, vel, np.sin(t), np.cos(t)], dtype=np.float32)
        obs_tensor = torch.tensor(obs_aug, dtype=torch.float32).unsqueeze(0)
        action = policy.get_action(obs_tensor)
        next_obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        obs = next_obs
        total_reward += reward
        t += 1
    print(total_reward)
env.close()


-200.0
-155.0
-162.0
-156.0
-160.0
-200.0
-200.0
-200.0
-155.0
-151.0
